# EvalGate score validation
Run this standalone demo with synthetic scores. The gate compares your candidate mean with a fixed baseline.

In [ ]:
"""EvalGate: deterministic executable reference pipeline."""

from __future__ import annotations


def run_pipeline(candidate_scores: list[float]) -> dict:
    if not candidate_scores:
        raise ValueError("Provide at least one candidate score.")
    if any(
        isinstance(score, bool)
        or not isinstance(score, (int, float))
        or not 0 <= score <= 1
        for score in candidate_scores
    ):
        raise ValueError("Candidate scores must be finite numbers between 0 and 1.")

    baseline = [0.86, 0.89, 0.91, 0.88]
    candidate = sum(candidate_scores) / len(candidate_scores)
    baseline_mean = sum(baseline) / len(baseline)
    delta = candidate - baseline_mean
    return {"candidate": round(candidate, 4), "baseline": round(baseline_mean, 4), "delta": round(delta, 4), "release_gate": "pass" if delta >= -0.01 else "blocked", "regressions": sum(s < 0.8 for s in candidate_scores)}



In [ ]:
assert run_pipeline([0.9, 0.91, 0.89])["release_gate"] == "pass"
assert run_pipeline([0.7, 0.72])["release_gate"] == "blocked"
for scores in ([], [float("nan")], [1.2]):
    try:
        run_pipeline(scores)
    except ValueError as error:
        print(error)
    else:
        raise AssertionError("Invalid scores were accepted")
